# Phase 5 - Feature Engineering & Feature Selection (Real Execution)

Source: `data/processed/cleaned_customer_churn.csv` (read-only).

All feature creation, validation and selection logic lives in `src/feature_engineering.py`;
this notebook calls the same functions the artifacts are saved from - no hard-coded metrics.

**Modeling boundary:** no model training, no scaler/encoder/imputer fitting in this phase.
The model-ready dataset keeps raw categorical values; encoding belongs to Phase 6.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.feature_engineering as fe

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
print("Cleaned file      :", fe.CLEAN_PATH)
print("Model-ready target:", fe.MODEL_READY_PATH)

Cleaned file      : data\processed\cleaned_customer_churn.csv
Model-ready target: data\processed\model_ready_data.csv


## 1. Load cleaned data

In [2]:
clean_hash_before = fe.sha256_of(fe.CLEAN_PATH)
print("Cleaned SHA-256 (before):", clean_hash_before)

df = fe.load_cleaned()
print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns (read-only)")
df.head()

Cleaned SHA-256 (before): 38e6f2dd278033a4617f880e8f11304c5fc6dce2c05ecef3f2a8bf48eeaa8996
Loaded 7043 rows x 21 columns (read-only)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 2. Feature inventory

Review of every original feature BEFORE engineering. 'Potential predictive value' is
computed from the data (churn-rate spread per category / Pearson r for numerics).
Nothing is removed just to reduce columns.

In [3]:
inventory = fe.feature_inventory(df)
display(inventory)

,Feature,Data type,Meaning,Unique values / range,Potential predictive value,Potential redundancy,Leakage risk,Final decision
0,customerID,str,Unique customer identifier,"7043 unique: 7590-VHVEG, 5575-GNVDE, 3668-QPYB...",None - unique per row,"None (identifier, not a predictor)",None - identifier,Keep in dataset for customer-level reporting o...
1,gender,str,Customer gender (Female / Male),"2 unique: Male, Female",Churn-rate spread 0.76 pts across categories (...,None structural; weak standalone signal - keep...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
2,SeniorCitizen,int64,"1 if customer is a senior citizen (65+), else 0","2 unique: 0, 1",Churn-rate spread 18.08 pts across categories ...,None,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
3,Partner,str,Customer has a spouse or partner (Yes / No),"2 unique: No, Yes",Churn-rate spread 13.29 pts across categories ...,None,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
4,Dependents,str,Customer has dependents such as children (Yes ...,"2 unique: No, Yes",Churn-rate spread 15.83 pts across categories ...,None,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
5,tenure,int64,Number of months the customer has been with th...,"range [0, 72], 73 unique values",Pearson r with churn = -0.3522,Overlaps engineered TenureGroup (binned copy);...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
6,PhoneService,str,Subscribes to home phone service (Yes / No),"2 unique: Yes, No",Churn-rate spread 1.78 pts across categories (...,Low variance (most customers Yes); prerequisit...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
7,MultipleLines,str,Multiple phone lines for phone customers (Yes ...,"3 unique: No, Yes, No phone service",Churn-rate spread 3.68 pts across categories (...,'No phone service' category is structurally de...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
8,InternetService,str,Internet service type (DSL / Fiber optic / No),"3 unique: Fiber optic, DSL, No",Churn-rate spread 34.49 pts across categories ...,'No' drives 'No internet service' in 6 add-on ...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...
9,OnlineSecurity,str,Online security add-on (Yes / No / No internet...,"3 unique: No, Yes, No internet service",Churn-rate spread 34.36 pts across categories ...,'No internet service' determined by InternetSe...,Low - pre-churn billing/service attribute,KEEP as candidate predictor (EDA supports sign...


## 3. EDA-driven feature selection

Categories used in this phase:

- **Candidate predictive features** - all 19 original predictors are kept (each shows measurable EDA signal or is retained so modeling can judge); 5 engineered features are added (sections 4-7).
- **Business-only fields** - `customerID` (customer-level reporting/predictions; excluded from the feature matrix).
- **Identifiers** - `customerID` (must not be a predictor).
- **Target** - `Churn` (never a predictor).

No variables are dropped preemptively; redundancy is documented and assessed at training time.

In [4]:
print(inventory[["Feature", "Potential predictive value", "Final decision"]].to_string(index=False))

         Feature                                                                                        Potential predictive value                                                                 Final decision
      customerID                                                                                             None - unique per row Keep in dataset for customer-level reporting only; EXCLUDE from feature matrix
          gender                               Churn-rate spread 0.76 pts across categories (26.16% for Male vs 26.92% for Female)    KEEP as candidate predictor (EDA supports signal; modeling phase finalizes)
   SeniorCitizen                                      Churn-rate spread 18.08 pts across categories (23.61% for 0 vs 41.68% for 1)    KEEP as candidate predictor (EDA supports signal; modeling phase finalizes)
         Partner                                   Churn-rate spread 13.29 pts across categories (19.66% for Yes vs 32.96% for No)    KEEP as candidate predicto

## 4. TenureGroup

Exact labels: `≤6 months`, `7–12 months`, `13–24 months`, `25+ months`.
`tenure = 0` falls in the first group. Validated against independent mask recomputation.

In [5]:
df, tg_checks = fe.add_tenure_group(df)
print("Counts:", tg_checks["counts"])
print("tenure=0 rows:", tg_checks["tenure_zero_in_first_group"], "(all in '≤6 months')")
print("Validations:", {k: v for k, v in tg_checks.items() if isinstance(v, bool)})

tg_churn = df.groupby("TenureGroup", observed=True)["Churn"].agg(customers="count", churned="sum")
tg_churn["churn_rate_pct"] = (tg_churn["churned"] / tg_churn["customers"] * 100).round(4)
display(tg_churn)

Counts: {'≤6 months': 1481, '7–12 months': 705, '13–24 months': 1024, '25+ months': 3833}
tenure=0 rows: 11 (all in '≤6 months')
Validations: {'no_missing': True, 'all_groups_nonempty': True, 'counts_sum_to_dataset': True, 'exactly_one_group_per_customer': True, 'counts_match_independent_masks': True}


,customers,churned,churn_rate_pct
TenureGroup,,,
≤6 months,1481,784,52.9372
7–12 months,705,253,35.8865
13–24 months,1024,294,28.7109
25+ months,3833,538,14.0360


## 5. MonthlyChargeBand

Method (reproducible, data-driven): quartiles of `MonthlyCharges` via `pd.qcut(q=4)`
computed **from this dataset**; boundaries are recorded below, not hard-coded.

In [6]:
df, mc_boundaries, mc_checks = fe.add_monthly_charge_band(df)
print("Quartile boundaries (from actual data):", mc_boundaries)
print("Counts:", mc_checks["counts"])
print("Validations:", mc_checks)

mb_churn = df.groupby("MonthlyChargeBand", observed=True)["Churn"].agg(customers="count", churned="sum")
mb_churn["churn_rate_pct"] = (mb_churn["churned"] / mb_churn["customers"] * 100).round(4)
display(mb_churn)

Quartile boundaries (from actual data): [18.25, 35.5, 70.35, 89.85, 118.75]
Counts: {'Low': 1762, 'Medium-Low': 1766, 'Medium-High': 1757, 'High': 1758}
Validations: {'no_missing': True, 'counts_sum_to_dataset': True, 'boundaries_strictly_increasing': True, 'counts': {'Low': 1762, 'Medium-Low': 1766, 'Medium-High': 1757, 'High': 1758}}


,customers,churned,churn_rate_pct
MonthlyChargeBand,,,
Low,1762,198,11.2372
Medium-Low,1766,434,24.5753
Medium-High,1757,659,37.5071
High,1758,578,32.8783


## 6. ServiceCount

Contributors (7 optional add-ons): `MultipleLines, OnlineSecurity, OnlineBackup,
DeviceProtection, TechSupport, StreamingTV, StreamingMovies`.
Treatment: `Yes=1`, `No=0`, `'No internet service'/'No phone service'=0`.
`PhoneService` (base voice, ~90% Yes) and `InternetService` (own categorical predictor)
are deliberately not counted - documented in the module.

In [7]:
df, sc_stats = fe.add_service_count(df)
print(f"min={sc_stats['min']}, max={sc_stats['max']}, mean={sc_stats['mean']}, "
      f"median={sc_stats['median']}, missing={sc_stats['missing']}")
print("ServiceCount==0 rows match customers with no 'Yes' add-on:",
      sc_stats["zero_count_matches_all_not_yes"])

display(df["ServiceCount"].value_counts().sort_index().rename("customers").to_frame())

sc_churn = df.groupby("ServiceCount", observed=True)["Churn"].agg(customers="count", churned="sum")
sc_churn["churn_rate_pct"] = (sc_churn["churned"] / sc_churn["customers"] * 100).round(4)
display(sc_churn)

min=0, max=7, mean=2.4597, median=2.0, missing=0
ServiceCount==0 rows match customers with no 'Yes' add-on: True


,customers
ServiceCount,
0,1667
1,1158
2,957
3,978
4,933
5,722
6,420
7,208


,customers,churned,churn_rate_pct
ServiceCount,,,
0,1667,346,20.7558
1,1158,399,34.4560
2,957,361,37.7220
3,978,306,31.2883
4,933,240,25.7235
5,722,157,21.7452
6,420,49,11.6667
7,208,11,5.2885


## 7. Additional engineered features (EDA-driven)

- **HasSecuritySupport** - EDA showed the protective add-ons have the strongest service-level
  association with retention (OnlineSecurity No 41.77% vs Yes 14.61%; TechSupport No 41.64% vs Yes 15.17%).
- **AvgMonthlyCharge** - charge-to-tenure relationship (TotalCharges/tenure); for `tenure=0`
  uses MonthlyCharges (no billing history yet). Separates current price from lifetime average spend.
- No interaction terms (Contract x Tenure etc.) are created: interactions are modeling-phase work;
  adding them now would increase dimensionality without evidence of need.

In [8]:
df = fe.add_has_security_support(df)
print("HasSecuritySupport counts:", df["HasSecuritySupport"].value_counts().to_dict())
hss = df.groupby("HasSecuritySupport", observed=True)["Churn"].agg(customers="count", churned="sum")
hss["churn_rate_pct"] = (hss["churned"] / hss["customers"] * 100).round(4)
display(hss)

df, amc_stats = fe.add_avg_monthly_charge(df)
print(f"AvgMonthlyCharge: min={amc_stats['min']}, max={amc_stats['max']}, mean={amc_stats['mean']}, "
      f"median={amc_stats['median']}, missing={amc_stats['missing']}")
print(f"tenure=0 handling verified ({amc_stats['tenure_zero_rows']} rows use MonthlyCharges):",
      amc_stats["tenure_zero_rows_use_monthlycharges"])
display(df[["tenure", "TotalCharges", "MonthlyCharges", "AvgMonthlyCharge"]].describe().round(4))

HasSecuritySupport counts: {0: 4079, 1: 2964}


,customers,churned,churn_rate_pct
HasSecuritySupport,,,
0,4079,1363,33.4151
1,2964,506,17.0715


AvgMonthlyCharge: min=13.775, max=121.4, mean=64.7629, median=70.3375, missing=0
tenure=0 handling verified (11 rows use MonthlyCharges): True


,tenure,TotalCharges,MonthlyCharges,AvgMonthlyCharge
count,7043.0000,7043.0000,7043.0000,7043.0000
mean,32.3711,2279.7343,64.7617,64.7629
std,24.5595,2266.7945,30.0900,30.1898
min,0.0000,0.0000,18.2500,13.7750
25%,9.0000,398.5500,35.5000,35.9352
50%,29.0000,1394.5500,70.3500,70.3375
75%,55.0000,3786.6000,89.8500,90.1742
max,72.0000,8684.8000,118.7500,121.4000


## 8. Leakage review

Every feature must be knowable BEFORE the customer churns. This dataset contains only
pre-churn billing/service attributes, so risk is Low everywhere; `customerID` (identifier)
and `Churn` (target) are excluded from predictors. Saved to
`outputs/reports/feature_leakage_review.csv` in section 12.

In [9]:
leak = fe.leakage_review(df)
display(leak)

,Feature,Leakage Risk,Included,Reason
0,customerID,None (identifier),No,Identifier; kept in the dataset for customer-l...
1,gender,Low,Yes,Pre-churn billing/service attribute: Customer ...
2,SeniorCitizen,Low,Yes,Pre-churn billing/service attribute: 1 if cust...
3,Partner,Low,Yes,Pre-churn billing/service attribute: Customer ...
4,Dependents,Low,Yes,Pre-churn billing/service attribute: Customer ...
5,tenure,Low,Yes,Pre-churn billing attribute: Number of months ...
6,TenureGroup,Low,Yes,Engineered from Engineered - known before chur...
7,PhoneService,Low,Yes,Pre-churn billing/service attribute: Subscribe...
8,MultipleLines,Low,Yes,Pre-churn billing/service attribute: Multiple ...
9,InternetService,Low,Yes,Pre-churn billing/service attribute: Internet ...


## 9. Redundancy analysis

Focus: `corr(tenure, TotalCharges) = 0.826` from EDA. Both are KEPT; assessment continues
in Phase 6 (VIF, regularized comparison, coefficient stability).

In [10]:
corr, red_notes = fe.redundancy_analysis(df)
display(corr)
for i, n in enumerate(red_notes, 1):
    print(f"{i}. {n}\n")

,tenure,MonthlyCharges,TotalCharges,ServiceCount,HasSecuritySupport,AvgMonthlyCharge
tenure,1.0000,0.2479,0.8262,0.5266,0.3518,0.2472
MonthlyCharges,0.2479,1.0000,0.6512,0.7730,0.4071,0.9962
TotalCharges,0.8262,0.6512,1.0000,0.7859,0.4820,0.6510
ServiceCount,0.5266,0.7730,0.7859,1.0000,0.6572,0.7711
HasSecuritySupport,0.3518,0.4071,0.4820,0.6572,1.0000,0.4065
AvgMonthlyCharge,0.2472,0.9962,0.6510,0.7711,0.4065,1.0000


1. corr(tenure, TotalCharges) = 0.8262: TotalCharges structurally accumulates as roughly tenure x average monthly price, so the pair is necessarily correlated - this is expected, not a data error.

2. Why keep BOTH: tenure measures relationship length; TotalCharges measures lifetime billing value. Two customers with equal tenure can have very different TotalCharges (price tier), and equal TotalCharges can hide very different tenures. Each carries information the other does not fully determine.

3. Multicollinearity concern: mainly affects linear models (inflated coefficient variance, unstable coefficients). Tree-based and boosted models are largely unaffected by pairwise collinearity.

4. Assessment plan (Phase 6, not now): compute VIF on the numeric block, compare regularized linear models with/without TotalCharges, and inspect coefficient stability. No feature is dropped preemptively.

5. corr(TotalCharges, AvgMonthlyCharge) = 0.6510, corr(tenure, AvgMonthlyCharge) = 0.2472: AvgMonth

## 10. Final feature list

In [11]:
print(f"Original predictors (candidates): {len(fe.ORIGINAL_PREDICTORS)}")
print(f"Engineered features (candidates): {len(fe.ENGINEERED_FEATURES)} -> {fe.ENGINEERED_FEATURES}")
print(f"Candidate model features       : {len(fe.ORIGINAL_PREDICTORS) + len(fe.ENGINEERED_FEATURES)}")
print(f"Identifier (kept, not predictor): {fe.ID_COL}")
print(f"Target                          : {fe.TARGET}")
print(f"Model-ready columns             : {len(fe.MODEL_READY_ORDER)}")

selection = fe.feature_selection_report(df, mc_boundaries)
display(selection)

Original predictors (candidates): 19
Engineered features (candidates): 5 -> ['TenureGroup', 'MonthlyChargeBand', 'ServiceCount', 'HasSecuritySupport', 'AvgMonthlyCharge']
Candidate model features       : 24
Identifier (kept, not predictor): customerID
Target                          : Churn
Model-ready columns             : 26


,Feature,Original/Engineered,Data Type,Included in Model,Reason,Leakage Risk,Redundancy Concern
0,customerID,Original,str,No (dataset only),Identifier for customer-level joins/predictions,None (identifier),"None (identifier, not a predictor)"
1,gender,Original,str,Yes,EDA shows signal: Churn-rate spread 0.76 pts a...,Low,None structural; weak standalone signal - keep...
2,SeniorCitizen,Original,int64,Yes,EDA shows signal: Churn-rate spread 18.08 pts ...,Low,None
3,Partner,Original,str,Yes,EDA shows signal: Churn-rate spread 13.29 pts ...,Low,None
4,Dependents,Original,str,Yes,EDA shows signal: Churn-rate spread 15.83 pts ...,Low,None
5,tenure,Original,int64,Yes,EDA shows signal: Pearson r with churn = -0.3522,Low,Overlaps engineered TenureGroup (binned copy);...
6,TenureGroup,Engineered,category,Yes,EDA: churn 52.94% (0-6m) vs 14.04% (25+m) - ea...,Low,Binned copy of tenure (exact duplication of in...
7,PhoneService,Original,str,Yes,EDA shows signal: Churn-rate spread 1.78 pts a...,Low,Low variance (most customers Yes); prerequisit...
8,MultipleLines,Original,str,Yes,EDA shows signal: Churn-rate spread 3.68 pts a...,Low,'No phone service' category is structurally de...
9,InternetService,Original,str,Yes,EDA shows signal: Churn-rate spread 34.49 pts ...,Low,'No' drives 'No internet service' in 6 add-on ...


## 11. Validation

`validate_model_ready` fails loudly (ValueError) on any violation.

In [12]:
for r in fe.validate_model_ready(df):
    print(r)

PASS  row count == 7043  (7043)
PASS  required columns present, nothing extra  (26 columns)
PASS  customerID unique / no duplicates
PASS  no missing values anywhere
PASS  Churn in {0,1}
PASS  TenureGroup labels exact  (['13–24 months', '25+ months', '7–12 months', '≤6 months'])
PASS  TenureGroup group counts > 0  ({'25+ months': 3833, '≤6 months': 1481, '13–24 months': 1024, '7–12 months': 705})
PASS  tenure=0 in first group  (11 rows)
PASS  MonthlyChargeBand labels exact
PASS  ServiceCount in [0,7]  (min=0, max=7)
PASS  HasSecuritySupport in {0,1}
PASS  AvgMonthlyCharge positive  (range [13.7750, 121.4000])
PASS  no leakage features present


## 12. Save model-ready dataset (+ all report artifacts) and verify integrity

In [13]:
paths = fe.finalize_and_save(df, verbose=False)
for name, p in paths.items():
    print(f"Saved {name:12s}: {p}  (exists={p.exists()})")

clean_hash_after = fe.sha256_of(fe.CLEAN_PATH)
print("\nCleaned SHA-256 (before):", clean_hash_before)
print("Cleaned SHA-256 (after) :", clean_hash_after)
assert clean_hash_before == clean_hash_after, "Cleaned dataset was modified!"
print("PASS: data/processed/cleaned_customer_churn.csv is byte-for-byte unchanged.")

Saved model_ready : data\processed\model_ready_data.csv  (exists=True)
Saved inventory   : outputs\reports\feature_inventory.csv  (exists=True)
Saved leakage     : outputs\reports\feature_leakage_review.csv  (exists=True)
Saved selection   : outputs\reports\feature_selection_report.csv  (exists=True)
Saved text_report : outputs\reports\feature_engineering_report.txt  (exists=True)

Cleaned SHA-256 (before): 38e6f2dd278033a4617f880e8f11304c5fc6dce2c05ecef3f2a8bf48eeaa8996
Cleaned SHA-256 (after) : 38e6f2dd278033a4617f880e8f11304c5fc6dce2c05ecef3f2a8bf48eeaa8996
PASS: data/processed/cleaned_customer_churn.csv is byte-for-byte unchanged.
